In [0]:
%sql
Use catalog adb_rtp;

insert into silver.daily_pricing_silver
select 
to_date(DATE_OF_PRICING,'dd/MM/yyyy'),
cast(ROW_ID as bigint),
STATE_NAME,
MARKET_NAME,
PRODUCTGROUP_NAME,
PRODUCT_NAME,
VARIETY,
ORIGIN,
cast(ARRIVAL_IN_TONNES as decimal(18,2)),
cast(MINIMUM_PRICE as decimal(36,2)),
cast(MAXIMUM_PRICE as decimal(36,2)),
cast(MODAL_PRICE as decimal(36,2)),
source_file_load_date,
current_timestamp(),
current_timestamp()
from adb_rtp.bronze.daily_pricing
where source_file_load_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'daily_pricing_silver' and process_status = 'complete')
;


--// always use the names of columns when using SQL on Databricks


--//Incremental load, Change Data Capture supports adding only the data that has been newly added to the table in previous stage

In [0]:
%sql
insert into adb_rtp.processrunlogs.deltalakehouse_process_runs(process_name,processed_file_table_date,process_status)
select 'daily_pricing_silver', max(source_file_load_date), 'complete' from silver.daily_pricing_silver;

Data Seems to be only pulling 01/01/2023 not the new data or changed values data, go through the mistakes in the first ingestion notebook